# Plan B - Datasource Report (Version 9)

Scans the entire Power BI / Fabric tenant using the **Scanner API** and produces
a complete report of **all datasources** across every dataset, with a focus on
identifying Databricks connections and their owner contact details.

**Version 9 changes vs Version 8:**
- **Scanner submission fixed** — scan options stay in the `SCAN_URL` query string,
  and the POST body only sends `{"workspaces": [...]}`.
- **Fallback datasource retrieval** — when scanner `datasourceUsages` cannot be
  resolved to detailed instances, the notebook calls
  `GET /admin/datasets/{id}/datasources` with retry/backoff for throttling.
- **Stronger datasource ID matching** — datasource instances are indexed by all
  supported ID fields (`datasourceId`, `datasourceInstanceId`, `id`) to reduce
  cross-key lookup mismatches.
- **Optional scan diagnostics cell** — quick raw-structure inspection for a
  sample workspace with datasets.
- Results written to **three Lakehouse Delta tables**:
  - `all_datasources` — every datasource found in the tenant
  - `databricks_all_connections` — Databricks subset
  - `databricks_personal_connections` — Databricks NOT on the VNet Gateway

**Prerequisites:**
- Enable **"Enhance admin APIs responses with detailed metadata"** in the
  Power BI tenant admin settings so the Scanner API returns full datasource
  instance details (type, connectionDetails, gatewayId).
- Fabric notebook with appropriate admin permissions.
- Default Lakehouse attached to the notebook.


In [ ]:
import json
import time
import pandas as pd
import requests as http_requests

# Spark session is available automatically in Fabric notebooks
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type":  "application/json",
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

# ── Configuration flags ──────────────────────────────────────────────
INCLUDE_PERSONAL_WORKSPACES = False   # Set True to also scan personal workspaces

# ── Known VNet Gateway IDs ── update this set with your tenant's VNet GW IDs
KNOWN_VNET_GATEWAY_IDS = {
    "3e481240-199c-402f-8fe3-ae5ef09b1ab4",
}

# ── Lakehouse table names (written to the default attached Lakehouse) ──
TABLE_ALL_DATASOURCES       = "all_datasources"
TABLE_DATABRICKS_ALL        = "databricks_all_connections"
TABLE_DATABRICKS_PERSONAL   = "databricks_personal_connections"

print("Setup complete.")
print(f"Include personal workspaces: {INCLUDE_PERSONAL_WORKSPACES}")

In [ ]:
# Fetch capacities
print("Fetching capacities...")
cap_map = {}

try:
    cap_resp = http_requests.get(f"{PBI_API}/admin/capacities", headers=HEADERS)
    if cap_resp.status_code == 200:
        for cap in cap_resp.json().get("value", []):
            cap_map[cap["id"]] = {
                "capacity_name": cap.get("displayName", ""),
                "capacity_sku":  cap.get("sku", ""),
            }
        print(f"{len(cap_map)} capacities loaded.")
    else:
        print(f"Status {cap_resp.status_code}: {cap_resp.text[:300]}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Fetch workspaces (with optional personal-workspace filter)
print("Fetching workspaces...")
raw_workspaces = []
top = 5000
skip = 0

while True:
    ws_resp = http_requests.get(
        f"{PBI_API}/admin/groups?$top={top}&$skip={skip}",
        headers=HEADERS,
    )
    if ws_resp.status_code == 429:
        retry = int(ws_resp.headers.get("Retry-After", 30))
        print(f"Rate limited. Sleeping {retry}s...")
        time.sleep(retry)
        continue
    if ws_resp.status_code != 200:
        print(f"Status {ws_resp.status_code}: {ws_resp.text[:300]}")
        break
    batch = ws_resp.json().get("value", [])
    if not batch:
        break
    raw_workspaces.extend(batch)
    if len(batch) < top:
        break
    skip += top

print(f"Total workspaces fetched: {len(raw_workspaces)}")

# ── Filter personal workspaces ──
if INCLUDE_PERSONAL_WORKSPACES:
    all_workspaces = raw_workspaces
    print("Including personal workspaces (flag is True).")
else:
    all_workspaces = [
        ws for ws in raw_workspaces
        if ws.get("type", "") != "PersonalGroup"
    ]
    excluded = len(raw_workspaces) - len(all_workspaces)
    print(f"Excluded {excluded} personal workspaces (type=PersonalGroup).")

print(f"Workspaces to scan: {len(all_workspaces)}")

ws_map = {}
for ws in all_workspaces:
    ws_map[ws["id"]] = {
        "workspace_name": ws.get("name", ""),
        "capacity_id":    ws.get("capacityId", ""),
    }

In [ ]:
# ── Concurrent batch scan using Scanner API ──────────────────────────
# Submits up to MAX_CONCURRENT scan requests in parallel, then polls
# them all, fetching results as each one completes.

BATCH_SIZE     = 100
MAX_CONCURRENT = 16     # API-enforced ceiling
POLL_INTERVAL  = 3      # seconds between poll sweeps
MAX_POLL_TIME  = 300    # per-scan timeout
SUBMIT_DELAY   = 1      # seconds between individual submissions

ws_ids = [ws["id"] for ws in all_workspaces]
total_batches = (len(ws_ids) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Scanning {len(ws_ids)} workspaces in {total_batches} batches "
      f"(up to {MAX_CONCURRENT} concurrent)...")

all_scan_workspaces = []
scan_errors = []

# NOTE: datasourceDetails=True + "Enhance admin APIs responses with
# detailed metadata" tenant setting gives full datasource info.
SCAN_URL = (
    f"{PBI_API}/admin/workspaces/getInfo"
    f"?datasourceDetails=True"
    f"&getArtifactUsers=True"
    f"&lineage=True"
    f"&datasetSchema=False"
    f"&datasetExpressions=False"
)

# Build list of batch ID lists
batches = []
for i in range(total_batches):
    start = i * BATCH_SIZE
    batches.append(ws_ids[start : start + BATCH_SIZE])

# ── Helper functions ─────────────────────────────────────────────────
def _submit_scan(batch_ids):
    """POST getInfo and return scan_id, or None on failure."""
    body = {"workspaces": batch_ids}
    for attempt in range(3):
        try:
            resp = http_requests.post(SCAN_URL, headers=HEADERS, json=body)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 30))
                print(f"    Rate limited on submit (attempt {attempt+1}). "
                      f"Sleeping {wait}s...")
                time.sleep(wait)
                continue
            if resp.status_code in (200, 202):
                info = resp.json()
                return info.get("id", info.get("scanId", ""))
            print(f"    Submit failed ({resp.status_code}): "
                  f"{resp.text[:200]}")
            return None
        except Exception as exc:
            print(f"    Submit error: {exc}")
            if attempt < 2:
                time.sleep(5)
    return None


def _fetch_results(scan_id):
    """GET scanResult and return list of workspace dicts."""
    for attempt in range(3):
        try:
            resp = http_requests.get(
                f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
                headers=HEADERS,
            )
            if resp.status_code == 200:
                return resp.json().get("workspaces", [])
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 30))
                time.sleep(wait)
                continue
            print(f"    Result fetch failed ({resp.status_code})")
            return []
        except Exception as exc:
            print(f"    Result error: {exc}")
            if attempt < 2:
                time.sleep(5)
    return []


# ── Main loop: submit → poll → collect ───────────────────────────────
# pending maps scan_id → {"batch_num": int, "submitted_at": float}
pending = {}
batch_idx = 0
completed_count = 0

while batch_idx < total_batches or pending:
    # ── Submit new scans up to MAX_CONCURRENT ──
    while batch_idx < total_batches and len(pending) < MAX_CONCURRENT:
        scan_id = _submit_scan(batches[batch_idx])
        if scan_id:
            pending[scan_id] = {
                "batch_num": batch_idx,
                "submitted_at": time.time(),
            }
        else:
            scan_errors.append({"batch": batch_idx, "error": "submit failed"})
        batch_idx += 1
        time.sleep(SUBMIT_DELAY)

    if not pending:
        break

    # ── Poll every pending scan ──
    time.sleep(POLL_INTERVAL)
    done_ids = []
    for scan_id, meta in list(pending.items()):
        elapsed = time.time() - meta["submitted_at"]
        try:
            resp = http_requests.get(
                f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
                headers=HEADERS,
            )
            if resp.status_code == 200:
                status = resp.json().get("status", "").lower()
                if status == "succeeded":
                    ws_results = _fetch_results(scan_id)
                    all_scan_workspaces.extend(ws_results)
                    completed_count += 1
                    done_ids.append(scan_id)
                    if completed_count % 10 == 0 or completed_count == total_batches:
                        print(f"  Progress: {completed_count}/{total_batches} "
                              f"batches complete "
                              f"({len(all_scan_workspaces)} workspaces)")
                elif status == "failed":
                    scan_errors.append({"batch": meta["batch_num"],
                                        "status": "failed"})
                    done_ids.append(scan_id)
            elif resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 15))
                time.sleep(wait)
        except Exception:
            pass

        # Timeout guard
        if elapsed > MAX_POLL_TIME and scan_id not in done_ids:
            scan_errors.append({"batch": meta["batch_num"],
                                "status": "timeout"})
            done_ids.append(scan_id)

    for sid in done_ids:
        pending.pop(sid, None)

print(f"\nScan complete. {len(all_scan_workspaces)} workspaces scanned.")
print(f"Completed batches: {completed_count}/{total_batches}")
if scan_errors:
    print(f"{len(scan_errors)} batch errors.")

In [ ]:
# ── Optional diagnostic: inspect raw scan structure ───────────────────
# Run this cell only when troubleshooting missing datasource details.

sample_ws = None
for ws in all_scan_workspaces:
    if ws.get("datasets"):
        sample_ws = ws
        break

if not sample_ws and all_scan_workspaces:
    sample_ws = all_scan_workspaces[0]

if not sample_ws:
    print("all_scan_workspaces is empty — no scan data available.")
else:
    print("Sample workspace:", sample_ws.get("name", "(unnamed)"))
    print("Top-level keys:", list(sample_ws.keys()))
    print("workspace-level datasourceInstances:",
          len(sample_ws.get("datasourceInstances", [])))

    datasets = sample_ws.get("datasets", [])
    print("datasets count:", len(datasets))
    if datasets:
        ds0 = datasets[0]
        print("dataset keys:", list(ds0.keys()))
        for key in ("datasourceUsages", "datasourceInstances", "datasources"):
            print(f"  dataset['{key}'] count:", len(ds0.get(key, [])))


In [ ]:
# ── Parse ALL datasources from scan results ─────────────────────────
# Every datasource for every dataset is captured.  An 'is_databricks'
# flag marks Databricks rows so you can filter later.
# Falls back to per-dataset Admin API when scan usages have no details.

print("Parsing scan results...\n")
results = []
unresolved_by_dataset = {}


def _build_row(ctx, src, src_id):
    """Build a result dict from a datasource dict."""
    ds_type  = src.get("datasourceType", "(unknown)")
    conn_raw = src.get("connectionDetails", {})
    conn_str = json.dumps(conn_raw) if conn_raw else "{}"
    gw_id    = src.get("gatewayId", "")

    # Databricks classification
    is_dbx = False
    if ds_type.lower() in ("databricks", "azuredatabricks"):
        is_dbx = True
    elif ds_type.lower() == "extension" and "databricks" in conn_str.lower():
        is_dbx = True

    # Gateway classification
    if not gw_id:
        gw_class = "No Gateway"
    elif gw_id in KNOWN_VNET_GATEWAY_IDS:
        gw_class = "VNet Gateway"
    else:
        gw_class = "Personal Cloud Gateway"

    # Parse host / httpPath from connectionDetails
    host_val = (
        conn_raw.get("host")
        or conn_raw.get("server")
        or conn_raw.get("path", "")
    )
    http_path_val = (
        conn_raw.get("httpPath")
        or conn_raw.get("database", "")
    )

    # Some responses wrap JSON inside the host string
    if isinstance(host_val, str) and host_val.strip().startswith("{"):
        try:
            parsed = json.loads(host_val)
            host_val = parsed.get("host") or parsed.get("server", host_val)
            if not http_path_val:
                http_path_val = parsed.get("httpPath", "")
        except Exception:
            pass

    return {
        "capacity_id":        ctx["capacity_id"],
        "capacity_name":      ctx["capacity_name"],
        "capacity_sku":       ctx["capacity_sku"],
        "workspace_id":       ctx["workspace_id"],
        "workspace_name":     ctx["workspace_name"],
        "dataset_id":         ctx["dataset_id"],
        "dataset_name":       ctx["dataset_name"],
        "datasource_id":      src_id,
        "datasource_type":    ds_type,
        "is_databricks":      is_dbx,
        "gateway_id":         gw_id,
        "gateway_class":      gw_class,
        "host":               host_val,
        "http_path":          http_path_val,
        "connection_details": conn_str,
        "owner_name":         ctx["configured_by"],
        "owner_email":        ctx["configured_by"],
    }


def _placeholder_row(ctx, inst_id):
    return {
        "capacity_id":        ctx["capacity_id"],
        "capacity_name":      ctx["capacity_name"],
        "capacity_sku":       ctx["capacity_sku"],
        "workspace_id":       ctx["workspace_id"],
        "workspace_name":     ctx["workspace_name"],
        "dataset_id":         ctx["dataset_id"],
        "dataset_name":       ctx["dataset_name"],
        "datasource_id":      inst_id,
        "datasource_type":    "(scan has no details)",
        "is_databricks":      False,
        "gateway_id":         "",
        "gateway_class":      "(unknown)",
        "host":               "",
        "http_path":          "",
        "connection_details": "",
        "owner_name":         ctx["configured_by"],
        "owner_email":        ctx["configured_by"],
    }


FALLBACK_MAX_RETRIES = 5
FALLBACK_INITIAL_WAIT = 5
FALLBACK_MAX_BACKOFF = 60


def _fetch_dataset_datasources(dataset_id):
    """Fallback API: GET /admin/datasets/{id}/datasources with retry/backoff."""
    url = f"{PBI_API}/admin/datasets/{dataset_id}/datasources"
    wait_seconds = FALLBACK_INITIAL_WAIT
    for attempt in range(FALLBACK_MAX_RETRIES):
        try:
            resp = http_requests.get(url, headers=HEADERS)
            if resp.status_code == 200:
                return resp.json().get("value", [])
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", wait_seconds))
                time.sleep(retry_after)
                continue
            return []
        except Exception:
            if attempt < FALLBACK_MAX_RETRIES - 1:
                time.sleep(wait_seconds)
                wait_seconds = min(wait_seconds * 2, FALLBACK_MAX_BACKOFF)
    return []


for ws_data in all_scan_workspaces:
    ws_id   = ws_data.get("id", "")
    ws_name = ws_data.get("name", "")

    capacity_id = ws_data.get("capacityId", "")
    if not capacity_id:
        capacity_id = ws_map.get(ws_id, {}).get("capacity_id", "")
    cap_info      = cap_map.get(capacity_id, {})
    capacity_name = cap_info.get("capacity_name", "")
    capacity_sku  = cap_info.get("capacity_sku", "")

    # Build datasource-instance lookup and index by all possible IDs.
    ds_instance_map = {}

    def _index_instance(inst):
        for id_key in ("datasourceId", "datasourceInstanceId", "id"):
            inst_id = inst.get(id_key, "")
            if inst_id:
                ds_instance_map[inst_id] = inst

    for inst in ws_data.get("datasourceInstances", []):
        _index_instance(inst)

    for dataset in ws_data.get("datasets", []):
        for key in ("datasourceInstances", "datasources"):
            for inst in dataset.get(key, []):
                _index_instance(inst)

    # Walk every dataset and collect datasource rows.
    for dataset in ws_data.get("datasets", []):
        dataset_id    = dataset.get("id", "")
        dataset_name  = dataset.get("name", "")
        configured_by = dataset.get("configuredBy", "")

        ctx = {
            "capacity_id": capacity_id,
            "capacity_name": capacity_name,
            "capacity_sku": capacity_sku,
            "workspace_id": ws_id,
            "workspace_name": ws_name,
            "dataset_id": dataset_id,
            "dataset_name": dataset_name,
            "configured_by": configured_by,
        }

        ds_key = (ws_id, dataset_id)
        usage_seen_ids = set()
        recorded_ids = set()

        # Method 1: datasourceUsages -> ds_instance_map
        for usage in dataset.get("datasourceUsages", []):
            usage_ids = [
                usage.get("datasourceInstanceId", ""),
                usage.get("datasourceId", ""),
                usage.get("id", ""),
            ]
            inst_id = next((x for x in usage_ids if x), "")
            if not inst_id or inst_id in usage_seen_ids:
                continue

            usage_seen_ids.add(inst_id)
            src = ds_instance_map.get(inst_id)
            if src:
                results.append(_build_row(ctx, src, inst_id))
                recorded_ids.add(inst_id)
                continue

            unresolved = unresolved_by_dataset.setdefault(
                ds_key,
                {
                    "dataset_id": dataset_id,
                    "context": ctx,
                    "missing_ids": set(),
                    "recorded_ids": set(),
                },
            )
            for uid in usage_ids:
                if uid:
                    unresolved["missing_ids"].add(uid)

        # Method 2: inline datasources / datasourceInstances
        for key in ("datasources", "datasourceInstances"):
            for src in dataset.get(key, []):
                src_id = (
                    src.get("datasourceId")
                    or src.get("datasourceInstanceId")
                    or src.get("id", "")
                )
                if not src_id or src_id in recorded_ids:
                    continue
                recorded_ids.add(src_id)
                results.append(_build_row(ctx, src, src_id))

        if ds_key in unresolved_by_dataset:
            unresolved_by_dataset[ds_key]["recorded_ids"].update(recorded_ids)
            unresolved_by_dataset[ds_key]["missing_ids"].difference_update(recorded_ids)

# Fallback for unresolved scanner usages.
fallback_dataset_count = 0
fallback_rows = 0

for item in unresolved_by_dataset.values():
    missing_ids = item["missing_ids"]
    if not missing_ids:
        continue

    dataset_id = item["dataset_id"]
    ctx = item["context"]
    recorded_ids = item["recorded_ids"]

    fallback_sources = _fetch_dataset_datasources(dataset_id)
    if fallback_sources:
        fallback_dataset_count += 1

    for src in fallback_sources:
        src_ids = [
            src.get("datasourceId", ""),
            src.get("datasourceInstanceId", ""),
            src.get("id", ""),
        ]
        src_id = next((x for x in src_ids if x), "")

        if src_id and src_id in recorded_ids:
            continue

        if not src_id:
            if len(missing_ids) == 1:
                src_id = next(iter(missing_ids), "")
            else:
                print(f"    Warning: fallback datasource for dataset {dataset_id} has no ID.")

        if src_id:
            recorded_ids.add(src_id)
            missing_ids.discard(src_id)
        for sid in src_ids:
            if sid:
                missing_ids.discard(sid)

        results.append(_build_row(ctx, src, src_id))
        fallback_rows += 1

    for unresolved_id in sorted(missing_ids):
        results.append(_placeholder_row(ctx, unresolved_id))

# Summary
total_rows  = len(results)
detail_rows = sum(1 for r in results if r["datasource_type"] != "(scan has no details)")
no_detail   = total_rows - detail_rows
dbx_rows    = sum(1 for r in results if r["is_databricks"])

print(f"Total datasource rows:   {total_rows}")
print(f"  With full details:     {detail_rows}")
print(f"  Without details:       {no_detail}")
print(f"  Databricks:            {dbx_rows}")
print(f"  Fallback datasets:     {fallback_dataset_count}")
print(f"  Fallback rows added:   {fallback_rows}")

if no_detail > 0:
    print("\n⚠  Some datasource instances still lack details.")
    print("   Make sure the tenant setting 'Enhance admin APIs responses")
    print("   with detailed metadata' is ENABLED, then re-run this notebook.")


In [ ]:
# ── Build DataFrames and write to Lakehouse Delta tables ─────────────
df_all = pd.DataFrame(results)

if df_all.empty:
    print("No datasources found. Nothing to write.")
else:
    # Convert is_databricks to string for Spark compatibility
    df_all["is_databricks"] = df_all["is_databricks"].map({True: "True", False: "False"})
    df_all = df_all.astype(str).fillna("")

    df_all = df_all.sort_values(
        by=["is_databricks", "gateway_class", "owner_email",
            "capacity_name", "workspace_name"],
        ascending=[False, False, True, True, True],
    ).reset_index(drop=True)

    df_dbx_all = df_all[df_all["is_databricks"] == "True"].copy()
    df_dbx_personal = df_dbx_all[
        df_dbx_all["gateway_class"] != "VNet Gateway"
    ].copy()

    # ── Summary ──────────────────────────────────────────────────────
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"\nAll datasources: {len(df_all)}")
    print(f"Databricks datasources: {len(df_dbx_all)}")
    print(f"Databricks not on VNet: {len(df_dbx_personal)}")

    print(f"\nDatasource types:")
    print(df_all.groupby("datasource_type").size().to_string())

    if not df_dbx_all.empty:
        print(f"\nDatabricks by gateway type:")
        print(df_dbx_all.groupby("gateway_class").size().to_string())

        print(f"\nDatabricks by capacity:")
        print(df_dbx_all.groupby(
            ["capacity_name", "capacity_sku", "gateway_class"]
        ).size().to_string())

    if not df_dbx_personal.empty:
        print(f"\nDatabricks non-VNet by owner:")
        print(df_dbx_personal.groupby(
            ["owner_email", "gateway_class"]
        ).size().to_string())
        print(f"\nUnique owners to contact: "
              f"{df_dbx_personal['owner_email'].nunique()}")

    # ── Write: all_datasources ───────────────────────────────────────
    spark_df = spark.createDataFrame(df_all)
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE_ALL_DATASOURCES)
    )
    print(f"\nWritten {len(df_all)} rows → '{TABLE_ALL_DATASOURCES}'")

    # ── Write: databricks_all_connections ────────────────────────────
    if not df_dbx_all.empty:
        spark_dbx = spark.createDataFrame(df_dbx_all)
        (
            spark_dbx.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TABLE_DATABRICKS_ALL)
        )
        print(f"Written {len(df_dbx_all)} rows → '{TABLE_DATABRICKS_ALL}'")
    else:
        print(f"No Databricks datasources — '{TABLE_DATABRICKS_ALL}' not updated.")

    # ── Write: databricks_personal_connections ───────────────────────
    if not df_dbx_personal.empty:
        spark_pers = spark.createDataFrame(df_dbx_personal)
        (
            spark_pers.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TABLE_DATABRICKS_PERSONAL)
        )
        print(f"Written {len(df_dbx_personal)} rows → "
              f"'{TABLE_DATABRICKS_PERSONAL}'")
    else:
        print(f"No personal/non-VNet Databricks connections — "
              f"'{TABLE_DATABRICKS_PERSONAL}' not updated.")

    print("\nDone. Tables are now queryable via the Lakehouse SQL endpoint.")

In [ ]:
# Optional: print email templates for Databricks dataset owners

if not df_all.empty and not df_dbx_personal.empty:
    print("=" * 70)
    print("EMAIL TEMPLATES")
    print("=" * 70)

    for owner_email, grp in df_dbx_personal.groupby("owner_email"):
        datasets_list = "\n".join([
            f"   - {row['dataset_name']}\n"
            f"          (workspace: {row['workspace_name']}"
            f", capacity: {row['capacity_name']})"
            for _, row in grp.iterrows()
        ])
        hosts = grp["host"].unique()
        paths = grp["http_path"].unique()

        print(f"\n{'─' * 60}")
        print(f"To: {owner_email}")
        print(f"Subject: Action Required - Migrate Databricks connection")
        print(f"")
        print(f"Hi {grp.iloc[0]['owner_name']},")
        print(f"")
        print(f"These datasets need migration to VNet Data Gateway:")
        print(f"")
        print(datasets_list)
        print(f"")
        print(f"Host(s): {', '.join(str(h) for h in hosts)}")
        print(f"Path(s): {', '.join(str(p) for p in paths)}")
        print(f"")
        print(f"Steps:")
        print(f"  1. Power BI > Settings > Manage connections and gateways")
        print(f"  2. Select VNet Data Gateway")
        print(f"  3. Add data source > Databricks > enter host + HTTP path")
        print(f"  4. Sign in with OAuth2")
        print(f"  5. Dataset settings > Gateway > select VNet datasource")
        print(f"  6. Refresh dataset")
        print(f"{'─' * 60}")
else:
    print("No personal Databricks connections to report.")